# How to Fine-Tune a model: From Hyperparameter Tuning Techniques to Validation Set Fine-Tuning

Welcome to Model Optimization. A model trained with default hyperparameters is rarely optimal. It might underfit the data (failing to capture the analytical signal) or overfit the data (memorizing the noise).

### The Analytical Pipeline:
1. **Environment Setup:** Importing data science and ML libraries.
2. **Data Splitting (Train/Val/Test):** Establishing strict mathematical boundaries to prevent data leakage during tuning.
3. **The Baseline:** Establishing a metric floor using default parameters.
4. **Automated Search:** Implementing `GridSearchCV` and `RandomizedSearchCV` to systematically map the hyperparameter space.
5. **Validation Set Fine-Tuning (Early Stopping):** Using a dedicated validation set to dynamically stop training the moment a model begins to overfit.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, run: !pip install scikit-learn xgboost pandas numpy

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb

# Set random seed for analytical reproducibility
np.random.seed(42)

print("Libraries imported successfully. Ready for Fine-Tuning.")

## Step 1: Data Splitting (The Three-Way Split)

In basic ML, you split data into Training and Testing. 

However, when Fine-Tuning, if you repeatedly test your hyperparameters on the Test set, your model will implicitly "memorize" the Test set. This is called **Data Leakage**.

To solve this, we create a three-way split:
1. **Training Set (60%):** Used by the algorithm to learn the parameters (weights).
2. **Validation Set (20%):** Used by the engineer to tune the *hyperparameters*.
3. **Test Set (20%):** Locked away in a vault. Used exactly *once* at the very end to evaluate true real-world performance.

In [ ]:
# Cell 4: Generating and Splitting the Data

# 1. Generate a complex synthetic classification dataset
X, y = make_classification(
    n_samples=2000, 
    n_features=20, 
    n_informative=15, 
    n_redundant=5, 
    random_state=42
)

# 2. First Split: Separate out the Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Second Split: Separate the remaining data into Training (75% of temp -> 60% of total) and Validation (25% of temp -> 20% of total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print("--- Analytical Data Split ---")
print(f"Training Set:   {X_train.shape[0]} samples (Learn Parameters)")
print(f"Validation Set: {X_val.shape[0]} samples (Tune Hyperparameters)")
print(f"Test Set:       {X_test.shape[0]} samples (Final Evaluation)")

## Step 2: Establishing a Baseline

Before we tune, we must establish a baseline. We will train a standard **Random Forest** using scikit-learn's default hyperparameters. If our tuning efforts cannot beat this score, we know our search space is flawed.

In [ ]:
# Cell 6: The Baseline Model

# Initialize model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)

# Train on the training data
baseline_model.fit(X_train, y_train)

# Evaluate on the validation data
baseline_preds = baseline_model.predict(X_val)
baseline_accuracy = accuracy_score(y_val, baseline_preds)

print(f"Baseline Validation Accuracy (Default Hyperparameters): {baseline_accuracy * 100:.2f}%")

## Step 3: Grid Search and Random Search

We will now systematically search for better hyperparameters. 

1. **Grid Search (`GridSearchCV`):** We define a dictionary of exact values. The algorithm will mathematically test every single possible permutation. It is exhaustive but computationally expensive.
2. **Random Search (`RandomizedSearchCV`):** We define a dictionary of values or distributions. The algorithm will randomly sample combinations for a set number of iterations. As seen in our interactive widget, this is often mathematically superior in high-dimensional spaces because it tests more unique values of the most important hyperparameter.

*Note: Cross-Validation (CV) internally splits the `X_temp` data into its own training/validation folds, ensuring robust evaluation.*

In [ ]:
# Cell 8: Automated Hyperparameter Tuning

# 1. Define the Hyperparameter Search Space
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 2. Random Search Implementation
print("Executing Randomized Search (Testing 20 random combinations)...")
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20, # Limit to 20 random combinations
    cv=3,      # 3-fold cross-validation
    scoring='accuracy',
    n_jobs=-1, # Use all available CPU cores
    random_state=42
)

# Fit the search on the combined temp data (Train + Val)
random_search.fit(X_temp, y_temp)

# Extract the best model
best_rf_model = random_search.best_estimator_

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters Found:\n{random_search.best_params_}")
print(f"Random Search CV Accuracy: {random_search.best_score_ * 100:.2f}%")

## Step 4: Validation Set Fine-Tuning (Early Stopping)

While searching grids is powerful, modern algorithms like **Gradient Boosting** (XGBoost, LightGBM) allow for dynamic fine-tuning during the training process itself. 

Instead of guessing the optimal `n_estimators` (number of trees), we set it to a massive number (e.g., 1000). We then pass our dedicated **Validation Set** into the training loop using an `eval_set`. 

After adding every tree, the algorithm tests itself on the validation set. If the validation loss stops improving for $N$ rounds, the algorithm mathematically concludes it has begun overfitting and **Early Stops**, effectively fine-tuning the exact optimal number of trees dynamically!

In [ ]:
# Cell 10: Early Stopping with XGBoost

print("Training XGBoost with dynamic Early Stopping...\n")

# Initialize XGBoost Classifier
# early_stopping_rounds=15 means: "Stop training if the validation accuracy hasn't improved in 15 consecutive trees"
xgb_model = xgb.XGBClassifier(
    n_estimators=1000, 
    learning_rate=0.05,
    max_depth=6,
    early_stopping_rounds=15, 
    eval_metric='logloss',
    random_state=42
)

# Fit the model, passing the strict Validation Set for dynamic evaluation
# verbose=50 prints the progress every 50 trees
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# Extract the dynamically discovered optimal number of trees
optimal_trees = xgb_model.best_ntree_limit

print("\n--- Early Stopping Results ---")
print(f"Training halted dynamically. The optimal number of trees was found to be: {optimal_trees}")

## Step 5: Final Evaluation

We have trained our baseline, tuned a Random Forest, and dynamically early-stopped an XGBoost model. 

Now, and *only* now, do we unlock the **Test Set** vault. We evaluate our best models on this completely unseen data to report our final, mathematically unbiased performance metrics.

In [ ]:
# Cell 12: The Unbiased Final Test

# 1. Evaluate Baseline Model
test_baseline = accuracy_score(y_test, baseline_model.predict(X_test))

# 2. Evaluate Tuned Random Forest
test_tuned_rf = accuracy_score(y_test, best_rf_model.predict(X_test))

# 3. Evaluate Fine-Tuned XGBoost
test_xgb = accuracy_score(y_test, xgb_model.predict(X_test))

print("=== FINAL TEST SET ACCURACY ===")
print(f"1. Baseline Random Forest:  {test_baseline * 100:.2f}%")
print(f"2. Random Search RF:        {test_tuned_rf * 100:.2f}%")
print(f"3. Early-Stopped XGBoost:   {test_xgb * 100:.2f}%")

print("\nAnalytical Conclusion: By systematically searching the hyperparameter space and dynamically preventing overfitting via the validation set, we successfully extracted higher performance from our algorithms without changing the underlying data.")